# Research Task - Local Transit Projects List #890
https://github.com/cal-itp/data-analyses/issues/890

Start working on a local (transit) projects list, perhaps by reaching out to people like Jeff Tumlin?
Amanda to start from LRTP lists, Christian to shadow/ask more detailed transit priority questions. "Across CA, how much would we save with transit priority implementation?"

Connected to Julian (student assistant) scope (analyze benefits from list)

Received more clarification. Starting with MPOs LRTP, identify projects that improve existing transit performance (upgrading something, improve ridership, etc). Then compare this short list to the remaining projects in LRTP (% of..., cost vs...). May need to look at previous LRTPs to see if same projects were mentioned previously to see if projects were completed or not.


## Next Steps
Start with reading in Amanda's data from gcs (list of all projects from MPO LRTPs)

In [1]:
import pandas as pd


In [11]:
my_df = pd.read_excel('gs://calitp-analytics-data/data-analyses/project_list/LRTP/all_LRTP_LOST.xlsx')

In [13]:
display(type(my_df))
display(my_df.shape)
display(list(my_df.columns))
display(my_df.head())

pandas.core.frame.DataFrame

(16276, 9)

['project_title',
 'lead_agency',
 'project_year',
 'project_description',
 'total_project_cost',
 'city',
 'county',
 'data_source',
 'notes']

,project_title,lead_agency,project_year,project_description,total_project_cost,city,county,data_source,notes
0,Carmel To Pebble Beach Bike/Ped Facility,Ambag,None,Construct Class I Or Class Ii Bike Facility.,86000.0,None,Santa Cruz,Ambag Lrtp,NaN
1,Sr 1 Carmel Corridor Between Carmel River Brid...,Ambag,None,Provide Accomodation For Bicyclists Along Stat...,500000.0,None,Santa Cruz,Ambag Lrtp,NaN
2,"Rio Road Traffic Calming, Pedestrian Access An...",Ambag,None,"Install Traffic Calming Devices, Enhance Visib...",250000.0,None,Santa Cruz,Ambag Lrtp,NaN
3,Eighth And San Antonio Avenues Class Ii Bike I...,Ambag,None,"Install Signs, Pavement Markings, Intersection...",80000.0,None,Santa Cruz,Ambag Lrtp,NaN
4,Pedestrian Pathway Behind Larson Field And Rio...,Ambag,None,Construct Pedestrian And Possible Bike Route A...,75000.0,None,Santa Cruz,Ambag Lrtp,NaN


In [14]:
my_df.data_source.value_counts()

Fresno Cog Lrtp    3147
Scag Lrtp          2952
Lost               1849
Sacog Lrtp         1601
Kern Cog Lrtp      1411
Scrtpa Lrtp        1066
Madera Ctc Lrtp     765
Stancog Lrtp        552
Slocog Lrtp         420
Sbcag Lrtp          419
Sandag Lrtp         416
Tcag Lrtp           337
Mtc Lrtp            282
Ambag Lrtp          280
Sjcog Lrtp          262
Bcag Lrtp           250
Mcagov Lrtp         108
Kcag Lrtp            84
Tmpo Lrtp            75
Name: data_source, dtype: int64

## Next
Find Amanda's code snippet of filtering the df by string values. specifically, filtering the projects that are trasnit 
related (either project name/desc/notes)

see these for reference:

https://github.com/cal-itp/data-analyses/blob/2b88d23f3895840cd8714f213ec6a4c1fc55c088/project_list/_categorizing_utils.py#L135

https://github.com/cal-itp/data-analyses/blob/main/project_list/_specific_list_utils.py#L44


keywords for transit
`TRANSIT = [
        "bus",
        "metro",
        "station",  # Station comes up a few times as a charging station and also as a train station
        "transit",
        "fare",
        "brt",
        "yarts",
        "railroad",
        "highway-rail",
        "streetcar",
        "mass transit",
        # , 'station' in description and 'charging station' not in description
`

In [15]:
transit_keywords = [
        "bus",
        "metro",
        "station",  # Station comes up a few times as a charging station and also as a train station
        "transit",
        "fare",
        "brt",
        "yarts",
        "railroad",
        "highway-rail",
        "streetcar",
        "mass transit",]
        # , 'station' in description and 'charging station' not in description

In [19]:
def lower_case(df, columns_to_search: list):
    """
    Lowercase and clean certain columns:
    """
    new_columns = []
    for i in columns_to_search:
        df[f"lower_case_{i}"] = (
            df[i]
            .str.lower()
            .fillna("none")
            .str.replace("-", "")
            .str.replace(".", "")
            .str.replace(":", "")
        )
        new_columns.append(f"lower_case_{i}")

    return df, new_columns

In [17]:
#ripping amanda's code for find_keywords

def find_keywords(df, columns_to_search: list, keywords_search: list):
    """
    Find keywords in certain columns
    """
    df2, lower_case_cols_list = lower_case(df, columns_to_search)

    keywords_search = f"({'|'.join(keywords_search)})"

    for i in lower_case_cols_list:
        df2[f"{i}_keyword_search"] = (
            df2[i].str.extract(keywords_search).fillna("keyword not found")
        )

    return df2

In [10]:
# ripping Amanda's code from LRTP project list 

def filter_projects(
    df,
    columns_to_search: list,
    keywords_search: list,
    file_name: str,
    save_to_gcs: bool = True,
):

    # Filter out for Cordon
    df = find_keywords(df, columns_to_search, keywords_search)
    df2 = (
        df[
            (df.lower_case_project_title_keyword_search != "keyword not found")
            | (df.lower_case_project_description_keyword_search != "keyword not found")
        ]
    ).reset_index(drop=True)

    # Delete out non HOV projects that were accidentally picked up
    projects_to_delete = [
        "SR 17 Corridor Congestion Relief in Los Gatos",
        "Interstate 380 Congestion Improvements",
    ]
    df2 = df2[~df2.project_title.isin(projects_to_delete)].reset_index(drop=True)

    # Change cases
    for i in ["project_title", "project_description"]:
        df2[i] = df2[i].str.title()

    # Drop invalid geometries
    gdf = df2[df2.geometry != None].reset_index(drop=True)
    gdf = gdf.set_geometry("geometry")
    gdf.geometry = gdf.geometry.set_crs("EPSG:4326")
    gdf = gdf[gdf.geometry.is_valid].reset_index(drop=True)
    gdf = gdf.fillna(gdf.dtypes.replace({"float64": 0.0, "object": "None"}))

    # One version that's a df
    columns_to_drop = ["lower_case_project_title", "lower_case_project_description"]
    df2 = df2.drop(columns=columns_to_drop + ["geometry"])
    df2 = df2.fillna(df.dtypes.replace({"float64": 0.0, "object": "None"}))

   

    return gdf, df2

In [21]:
# ripping Amanda's code from LRTP project list 

filter_projects(
    my_df,
    [
        "project_title",
        "project_description",
    ],
    transit_keywords,
    False,
)

/tmp/ipykernel_318/876034870.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df[i]


AttributeError: 'DataFrame' object has no attribute 'geometry'